# Advanced Activity: Custom MCP Client — Cat Shop

This notebook builds a custom MCP client that:
1. Connects to the Cat Shop MCP server over **Streamable HTTP**
2. Authenticates via **OAuth 2.1** (browser-based sign-in)
3. Orchestrates a **multi-step shopping flow** — browse → add to cart → checkout
4. Demonstrates the new **`instant_order`** one-shot tool
5. Compares the **MCP developer experience** to traditional REST API orchestration

**Prerequisites:** The Cat Shop server must be running locally.
```bash
cd 08_MCP && uv run uvicorn app.server:app --port 8000
```

## 1. Imports & Configuration

In [1]:
import asyncio
import json
import os
import webbrowser
from urllib.parse import parse_qs, urlsplit

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from mcp.client.auth import OAuthClientProvider, TokenStorage
from mcp.shared.auth import OAuthClientInformationFull, OAuthClientMetadata, OAuthToken
from pydantic import AnyUrl

load_dotenv()

# Must match the ISSUER_URL the server was started with.
# Running through ngrok? Add ISSUER_URL=https://your-tunnel.ngrok-free.app to .env
SERVER_URL    = os.getenv("ISSUER_URL", "http://localhost:8000").rstrip("/")
MCP_URL       = f"{SERVER_URL}/mcp"
CALLBACK_PORT = 8766          # distinct from the CLI client (8765)
MODEL         = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

SYSTEM_PROMPT = """
You are a helpful Cat Shop assistant. Use the provided MCP tools to answer
catalog and cart questions. Never invent product information.
Only call checkout when the user explicitly asks to place an order.
"""

print(f"Model : {MODEL}")
print(f"Server: {SERVER_URL}")

Model : gpt-4o-mini
Server: https://suspect-conjure-unify.ngrok-free.dev


## 2. OAuth Helpers

MCP's OAuth 2.1 flow requires two pieces:
- **`InMemoryTokenStorage`** — keeps the access/refresh token for the life of this kernel session.
- **`OAuthCallbackServer`** — a tiny HTTP server that listens on `localhost:8766` and catches the `?code=` redirect from the Cat Shop login page.

In [2]:
class InMemoryTokenStorage(TokenStorage):
    """Holds OAuth tokens for the duration of this notebook session."""

    def __init__(self) -> None:
        self._tokens: OAuthToken | None = None
        self._client_info: OAuthClientInformationFull | None = None

    async def get_tokens(self) -> OAuthToken | None:
        return self._tokens

    async def set_tokens(self, tokens: OAuthToken) -> None:
        self._tokens = tokens

    async def get_client_info(self) -> OAuthClientInformationFull | None:
        return self._client_info

    async def set_client_info(self, client_info: OAuthClientInformationFull) -> None:
        self._client_info = client_info


class OAuthCallbackServer:
    """Minimal HTTP server that receives the OAuth redirect and extracts the auth code."""

    def __init__(self, host: str = "127.0.0.1", port: int = 8766) -> None:
        self.host = host
        self.port = port
        self._server: asyncio.AbstractServer | None = None
        self._callback: asyncio.Future | None = None

    @property
    def callback_url(self) -> str:
        return f"http://{self.host}:{self.port}/callback"

    async def __aenter__(self) -> "OAuthCallbackServer":
        self._callback = asyncio.get_running_loop().create_future()
        self._server = await asyncio.start_server(self._handle_request, self.host, self.port)
        return self

    async def __aexit__(self, *_) -> None:
        if self._server:
            self._server.close()
            await self._server.wait_closed()

    async def wait_for_callback(self) -> tuple[str, str | None]:
        if self._callback is None:
            raise RuntimeError("Callback server not started.")
        return await self._callback

    async def _handle_request(self, reader: asyncio.StreamReader, writer: asyncio.StreamWriter) -> None:
        try:
            request_line = (await reader.readline()).decode().strip()
            parts = request_line.split(" ", 2)
            target = parts[1] if len(parts) >= 2 else "/"
            while await reader.readline() not in (b"", b"\r\n"):
                pass
            parsed = urlsplit(target)
            params = parse_qs(parsed.query)
            code  = params.get("code",  [None])[0]
            state = params.get("state", [None])[0]
            error = params.get("error", [None])[0]
            if parsed.path == "/callback" and code:
                if self._callback and not self._callback.done():
                    self._callback.set_result((code, state))
                body = "<h1>Signed in</h1><p>You can return to the notebook.</p>"
                status = "200 OK"
            elif error:
                if self._callback and not self._callback.done():
                    self._callback.set_exception(RuntimeError(f"OAuth error: {error}"))
                body = "<h1>Sign-in failed</h1>"
                status = "400 Bad Request"
            else:
                body = "<h1>Invalid callback</h1>"
                status = "400 Bad Request"
            response = (
                f"HTTP/1.1 {status}\r\nContent-Type: text/html\r\n"
                f"Content-Length: {len(body.encode())}\r\nConnection: close\r\n\r\n{body}"
            )
            writer.write(response.encode())
            await writer.drain()
        finally:
            writer.close()
            await writer.wait_closed()

print("OAuth helpers ready.")

OAuth helpers ready.


## 3. Connect to the MCP Server

Running this cell will:
1. Start the local OAuth callback server on port 8766
2. Initiate the OAuth flow — a browser tab will open to the Cat Shop login page
3. After you sign in, the token is stored in memory and the MCP client connects
4. All available tools are printed

> **Note:** If the browser doesn't open automatically, copy-paste the printed URL.

In [3]:
# Module-level state so subsequent cells can reuse the connected client
_token_storage = InMemoryTokenStorage()
_tools         = []
_agent         = None
_callback_ctx  = None


async def connect_to_server() -> None:
    global _tools, _agent, _callback_ctx

    # Clean up any leftover callback server from a previous run
    if _callback_ctx is not None:
        await _callback_ctx.__aexit__(None, None, None)
        _callback_ctx = None

    _callback_ctx = OAuthCallbackServer(port=CALLBACK_PORT)
    await _callback_ctx.__aenter__()

    async def redirect_handler(auth_url: str) -> None:
        print(f"\nSign in to Cat Shop:\n  {auth_url}\n")
        webbrowser.open(auth_url)

    oauth = OAuthClientProvider(
        server_url=SERVER_URL,
        client_metadata=OAuthClientMetadata(
            client_name="Cat Shop Notebook Client",
            redirect_uris=[AnyUrl(_callback_ctx.callback_url)],
            grant_types=["authorization_code", "refresh_token"],
            response_types=["code"],
            scope="read write",
        ),
        storage=_token_storage,
        redirect_handler=redirect_handler,
        callback_handler=_callback_ctx.wait_for_callback,
    )

    mcp_client = MultiServerMCPClient({
        "cat_shop": {
            "transport": "streamable_http",
            "url": MCP_URL,
            "auth": oauth,
        }
    })

    _tools = await mcp_client.get_tools()
    _agent = create_agent(
        ChatOpenAI(model=MODEL, temperature=0),
        _tools,
        system_prompt=SYSTEM_PROMPT,
    )

    print(f"Connected! {len(_tools)} tools loaded:")
    for t in _tools:
        print(f"  • {t.name}")


await connect_to_server()


Sign in to Cat Shop:
  https://suspect-conjure-unify.ngrok-free.dev/authorize?response_type=code&client_id=47d06365-6b75-4c3d-958c-f69c4b86538f&redirect_uri=http%3A%2F%2F127.0.0.1%3A8766%2Fcallback&state=r5jFmInXzpr45qstpINY_wR0cNz6X2USCzhnrPuDHrg&code_challenge=FqiIwQ-F_NBya9Bg_wOaUzcHupHamTIzgX_e5_kH56M&code_challenge_method=S256&resource=https%3A%2F%2Fsuspect-conjure-unify.ngrok-free.dev%2F&scope=read+write

Connected! 7 tools loaded:
  • list_products
  • get_product
  • add_to_cart
  • view_cart
  • remove_from_cart
  • instant_order
  • checkout


## 4. Inspect Tool Schemas

One of MCP's core DX wins: tools are **self-describing**. The LLM (and you) can inspect
the schema for any tool without reading API docs.

In [4]:
for tool in _tools:
    schema = tool.get_input_jsonschema()
    props  = schema.get("properties", {})
    params = ", ".join(
        f"{k}: {v.get('type', 'any')}"
        for k, v in props.items()
    ) or "(no params)"
    print(f"\n{'─'*60}")
    print(f"Tool   : {tool.name}({params})")
    print(f"Desc   : {tool.description[:120]}")


────────────────────────────────────────────────────────────
Tool   : list_products((no params))
Desc   : Browse the cat shop catalog. Optionally filter by category (toys, beds, food, furniture).

────────────────────────────────────────────────────────────
Tool   : get_product((no params))
Desc   : Get full details of a single product by its ID.

────────────────────────────────────────────────────────────
Tool   : add_to_cart((no params))
Desc   : Add a product to your shopping cart. If already in cart, quantity is increased.

────────────────────────────────────────────────────────────
Tool   : view_cart((no params))
Desc   : View everything in your shopping cart with quantities and totals.

────────────────────────────────────────────────────────────
Tool   : remove_from_cart((no params))
Desc   : Remove a product from your shopping cart.

────────────────────────────────────────────────────────────
Tool   : instant_order((no params))
Desc   : Buy a product immediately by name — f

## 5. Multi-Step Shopping Flow (Agent-Driven)

The agent receives a natural-language request and **autonomously decides** which tools
to call and in what order — browse catalog → pick a product → add to cart → checkout.
No orchestration code needed.

In [5]:
async def ask(prompt: str) -> str:
    result = await _agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
    content = result["messages"][-1].content
    return content if isinstance(content, str) else json.dumps(content, indent=2)


# --- Step 1: Browse the catalog ---
print("=" * 60)
print("STEP 1: Browse the catalog")
print("=" * 60)
response = await ask("Show me all available cat food products with prices.")
print(response)

STEP 1: Browse the catalog
Here are the available cat food products along with their prices:

1. **Salmon Treats**
   - Description: Freeze-dried wild salmon bites, 100g
   - Price: $7.99

2. **Tuna Crunchies**
   - Description: Crunchy tuna-flavored dental treats, 80g
   - Price: $5.99

Let me know if you need more information or want to add any of these to your cart!


In [6]:
# --- Step 2: Add to cart ---
print("=" * 60)
print("STEP 2: Add a product to cart")
print("=" * 60)
response = await ask("Add 2 of the cheapest cat food to my cart.")
print(response)

STEP 2: Add a product to cart
I have added 2 of the cheapest cat food, Tuna Crunchies, to your cart. If you need anything else, just let me know!


In [7]:
# --- Step 3: Review and checkout ---
print("=" * 60)
print("STEP 3: Checkout")
print("=" * 60)
response = await ask("Show my cart and then place the order.")
print(response)

STEP 3: Checkout
Your cart contained the following item:

- **Tuna Crunchies**: 2 units at $5.99 each (Subtotal: $11.98)

Your order has been successfully placed! 

**Order ID**: F609E782E7A69754  
**Total**: $11.98  
**Status**: Confirmed

Thank you, your cats will love their new goodies!


## 6. Instant Order — One Tool, Full Workflow

The `instant_order` tool collapses the entire browse → add-to-cart → checkout pipeline
into a single tool call. The agent (and the user) only needs to say what they want to buy.

In [8]:
print("=" * 60)
print("INSTANT ORDER: Natural language → confirmed order")
print("=" * 60)

response = await ask("Buy salmon treats for my cat.")
print(response)

INSTANT ORDER: Natural language → confirmed order
Your order for 1x Salmon Treats has been confirmed! The total is $7.99. Your cats will love it! If you need anything else, feel free to ask.


In [9]:
# Try ordering with quantity
response = await ask("Order 3 catnip mice right now.")
print(response)

Your order has been confirmed! 🎉

- **Product**: 3x Catnip Mouse
- **Subtotal**: $14.97
- **Order ID**: D0AA183AB47A58D2

Your cats will love it! If you need anything else, just let me know!


## 7. Direct Tool Invocation (No Agent)

MCP tools are LangChain `BaseTool` instances — you can call them directly with `.ainvoke()`
without going through the LLM. This is useful for scripting, testing, or building
deterministic pipelines on top of MCP.

In [10]:
# Build a quick name→tool lookup
tool_map = {t.name: t for t in _tools}


def _unwrap(result):
    """MCP tools return LangChain content blocks: [{"type":"text","text":"...json..."}].
    Unwrap to the actual Python value."""
    if isinstance(result, list):
        parsed = []
        for item in result:
            if isinstance(item, dict) and item.get("type") == "text":
                try:
                    parsed.append(json.loads(item["text"]))
                except (json.JSONDecodeError, KeyError):
                    parsed.append(item)
            else:
                parsed.append(item)
        return parsed[0] if len(parsed) == 1 else parsed
    if isinstance(result, str):
        try:
            return json.loads(result)
        except json.JSONDecodeError:
            return result
    return result


async def call_tool(name: str, **kwargs):
    result = await tool_map[name].ainvoke(kwargs)
    return _unwrap(result)


# --- Scripted multi-step flow (no LLM reasoning) ---
print("Browsing toys...")
toys = await call_tool("list_products", category="toys")
print(json.dumps(toys, indent=2))

if toys:
    first_toy = toys[0]
    print(f"\nAdding '{first_toy['name']}' to cart...")
    cart_result = await call_tool("add_to_cart", product_id=first_toy["id"], quantity=1)
    print(json.dumps(cart_result, indent=2))

    print("\nChecking out...")
    order = await call_tool("checkout")
    print(json.dumps(order, indent=2))

Browsing toys...
[
  {
    "id": 1,
    "name": "Whisker Wand",
    "description": "Interactive feather toy on a flexible wand",
    "price": 9.99,
    "category": "toys"
  },
  {
    "id": 2,
    "name": "Catnip Mouse",
    "description": "Organic catnip-stuffed plush mouse",
    "price": 4.99,
    "category": "toys"
  },
  {
    "id": 3,
    "name": "Laser Pointer Pro",
    "description": "Red-dot laser with adjustable patterns",
    "price": 12.99,
    "category": "toys"
  }
]

Adding 'Whisker Wand' to cart...
{
  "success": true,
  "message": "Added 1x Whisker Wand to your cart"
}

Checking out...
{
  "order_id": "686E27C2B6154CE8",
  "status": "confirmed",
  "items": [
    {
      "product_id": 1,
      "name": "Whisker Wand",
      "price": 9.99,
      "quantity": 1,
      "subtotal": 9.99
    }
  ],
  "total": 9.99,
  "message": "Order 686E27C2B6154CE8 confirmed! Thanks matt, your cats will love their new goodies!"
}


## 8. DX Comparison: MCP vs Traditional REST

Below we implement the same **"buy salmon treats"** flow two ways to highlight the
difference in developer experience.

### Approach A — MCP (what we just built)
```python
# The entire buy flow in one line:
await ask("Buy salmon treats for my cat.")
```
The LLM reads the tool schemas, decides to call `instant_order`, and returns a
natural-language confirmation. **Zero orchestration code.**

### Approach B — Traditional REST API
If the same capabilities were exposed as conventional REST endpoints, you'd need to
write all of this yourself:

In [11]:
import httpx


async def buy_with_rest_api(product_query: str, quantity: int = 1) -> dict:
    """
    Hypothetical REST equivalent of instant_order.
    Assumes the shop exposed: GET /api/products, POST /api/cart, POST /api/checkout.
    The developer must write every step and handle every error explicitly.
    """
    # In real life you'd obtain this token through your own OAuth client code.
    tokens = await _token_storage.get_tokens()
    if tokens is None:
        raise RuntimeError("Not authenticated — run the connect cell first.")
    headers = {"Authorization": f"Bearer {tokens.access_token}"}

    # NOTE: These endpoints don't exist on the Cat Shop server (it's MCP-only).
    # This code illustrates what you'd have to write against a conventional REST API.
    REST_BASE = f"{SERVER_URL}/api"  # hypothetical

    async with httpx.AsyncClient() as client:
        # Step 1: Fetch the full product catalog
        resp = await client.get(f"{REST_BASE}/products", headers=headers)
        resp.raise_for_status()
        all_products = resp.json()

        # Step 2: Filter manually — the API has no search endpoint
        matches = [p for p in all_products if product_query.lower() in p["name"].lower()]
        if not matches:
            raise ValueError(f"No products match '{product_query}'")
        if len(matches) > 1:
            names = [p["name"] for p in matches]
            raise ValueError(f"Ambiguous — multiple matches: {names}")
        product = matches[0]

        # Step 3: Add to cart
        resp = await client.post(
            f"{REST_BASE}/cart",
            json={"product_id": product["id"], "quantity": quantity},
            headers=headers,
        )
        resp.raise_for_status()

        # Step 4: Checkout
        resp = await client.post(f"{REST_BASE}/checkout", headers=headers)
        resp.raise_for_status()
        return resp.json()


print("""
REST version requires:
  • 4 explicit HTTP calls (vs 1 MCP tool call)
  • Manual search/filter logic the developer writes
  • Manual error handling for ambiguous matches
  • Manual auth header injection
  • Knowledge of every endpoint URL and request shape
  • No self-documentation — you must read the API docs

MCP version:
  • 1 natural language request to the agent
  • Tool schemas are self-describing — no docs needed
  • Auth is handled by the MCP OAuth layer
  • The LLM orchestrates and handles ambiguity with follow-up questions
""")


REST version requires:
  • 4 explicit HTTP calls (vs 1 MCP tool call)
  • Manual search/filter logic the developer writes
  • Manual error handling for ambiguous matches
  • Manual auth header injection
  • Knowledge of every endpoint URL and request shape
  • No self-documentation — you must read the API docs

MCP version:
  • 1 natural language request to the agent
  • Tool schemas are self-describing — no docs needed
  • Auth is handled by the MCP OAuth layer
  • The LLM orchestrates and handles ambiguity with follow-up questions



## 9. Side-by-Side Comparison

| Dimension | Traditional REST | MCP (this notebook) |
|---|---|---|
| **Tool discovery** | Read API docs manually | Auto-loaded from server at connect time |
| **Auth** | Per-API setup, manual header injection | Built into the protocol (OAuth 2.1) |
| **Orchestration** | Developer writes the flow | LLM reasons and calls tools as needed |
| **Ambiguity handling** | Developer codes it | LLM asks a follow-up question |
| **Adding a new operation** | New endpoint + docs + client code | Add `@mcp.tool()` — agent finds it automatically |
| **Multi-step workflows** | Write N HTTP calls in order | One natural language sentence |
| **Local dev** | Mock/stub the API | Use stdio transport — no network needed |
| **Production scaling** | Each client reimplements auth | Central OAuth server issues tokens to any client |

## 10. Cleanup

Shut down the local OAuth callback server when done.

In [12]:
if _callback_ctx is not None:
    await _callback_ctx.__aexit__(None, None, None)
    print("OAuth callback server stopped.")
else:
    print("Nothing to clean up.")

OAuth callback server stopped.
